In [1]:
import os

from urllib3 import request


In [20]:
from src.datascience import logger

In [2]:
%pwd

'C:\\Users\\anice\\PythonProject\\research'

In [3]:
%ls

 Le volume dans le lecteur C s'appelle Windows-SSD
 Le num�ro de s�rie du volume est B672-0010

 R�pertoire de C:\Users\anice\PythonProject\research

20/05/2026  05:49    <DIR>          .
20/05/2026  05:48    <DIR>          ..
20/05/2026  05:49               985 1_data_ingestion.ipynb
20/05/2026  05:43             1�220 research.ipynb
               2 fichier(s)            2�205 octets
               2 R�p(s)  18�569�392�128 octets libres


In [4]:
os.chdir("../")

In [5]:
%pwd


'C:\\Users\\anice\\PythonProject'

In [9]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [15]:
from src.datascience.utils.common import read_yaml, create_directories

In [7]:
from src.datascience.constants import *

In [10]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH, schema_filepath=SCHEMA_FILE_PATH):
        self.config =read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self)-> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(

                root_dir = config.root_dir,
            source_URL= config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir

        )
        return data_ingestion_config



In [18]:
from urllib import request

In [13]:
import zipfile


class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(url=self.config.source_URL,
            filename=self.config.local_data_file)
            logger.info(f"Downloaded file {filename}")
        else:
            logger.info(f"File {self.config.local_data_file} already exists")

    def extract_zip_file(self):
        unzipp_path = self.config.unzip_dir
        os.makedirs(unzipp_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file) as zf:
            zf.extractall(unzipp_path)




In [21]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e




[2026-05-20 06:14:32,657: INFO: common] yaml file: config\config.yaml loaded successfully]
[2026-05-20 06:14:32,659: INFO: common] yaml file: params.yaml loaded successfully]
[2026-05-20 06:14:32,661: INFO: common] yaml file: schema.yaml loaded successfully]
[2026-05-20 06:14:32,662: INFO: common] Created directory: artifacts]
[2026-05-20 06:14:32,663: INFO: common] Created directory: artifacts/data_ingestion]
[2026-05-20 06:14:32,663: INFO: 1458085515] File artifacts/data_ingestion/data.zip already exists]
